In [ ]:
import pandas as pd

# Optional helper for adding a random unique numeric ID column to a spreadsheet.
# Configure the source and destination paths through your local environment or ad hoc variables,
# not inside the committed notebook.

In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
model_name = os.getenv("AZURE_MODEL_NAME")
deployment = os.getenv("AZURE_DEPLOYMENT")

subscription_key = os.getenv("AZURE_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_API_VERSION")

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_completion_tokens=16384,
    model=deployment
)

print(response.choices[0].message.content)

In [ ]:
from project_config import (
    BIAS_NOTES_CLEANED_CSV,
    BIAS_NOTES_OUTPUT_DIR,
    BIAS_NOTES_PROMPT_PATH,
)

# --- CONFIG (shared Azure pipeline) ---
INPUT_CSV = BIAS_NOTES_CLEANED_CSV
OUTPUT_DIR = BIAS_NOTES_OUTPUT_DIR
OUTPUT_PREFIX = "bias_flagged"
PROMPT_PATH = BIAS_NOTES_PROMPT_PATH

N_ROWS_TO_PROCESS = 100
RANDOM_SELECTION = True
RANDOM_SEED = None

CHUNK_CHAR_LIMIT = 2800
MAX_RETRIES = 5
SLEEP_BASE_SEC = 1.4
PARALLEL_CALLS = 4
GLOBAL_MAX_CALLS = 8
TEMPERATURE = 0  # use 1 for GPT-5-family models if required

if not INPUT_CSV:
    raise RuntimeError(
        "Set BIAS_NOTES_CLEANED_CSV in .env before running this notebook."
    )
if not OUTPUT_DIR:
    raise RuntimeError(
        "Set BIAS_NOTES_OUTPUT_DIR in .env before running this notebook."
    )
if not PROMPT_PATH:
    raise RuntimeError(
        "Set BIAS_NOTES_PROMPT_PATH in .env before running this notebook."
    )

In [ ]:
# Shared Azure bias-assessment engine
import os
import time

from dotenv import load_dotenv
from openai import AzureOpenAI

from bias_pipeline import (
    AzureBiasPipeline,
    AzureBiasPipelineConfig,
    generate_output_path,
    write_output_bundle,
)

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_API_VERSION")
model_for_api = os.getenv("AZURE_DEPLOYMENT") or os.getenv("AZURE_MODEL_NAME")

if not all([endpoint, subscription_key, api_version, model_for_api]):
    raise RuntimeError(
        "Missing Azure OpenAI configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_SUBSCRIPTION_KEY, AZURE_API_VERSION, and AZURE_DEPLOYMENT/AZURE_MODEL_NAME."
    )

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

output_csv = globals().get("OUTPUT_CSV")
if not output_csv:
    output_csv = generate_output_path(OUTPUT_DIR, OUTPUT_PREFIX)

df_full = pd.read_csv(INPUT_CSV)
if "note_text" not in df_full.columns:
    raise ValueError("Input CSV must contain a 'note_text' column.")

total_rows = len(df_full)
n_to_process = (
    total_rows if N_ROWS_TO_PROCESS is None else min(N_ROWS_TO_PROCESS, total_rows)
)

if RANDOM_SELECTION and n_to_process < total_rows:
    if RANDOM_SEED is not None:
        df = df_full.sample(n=n_to_process, random_state=RANDOM_SEED).reset_index(
            drop=True
        )
    else:
        df = df_full.sample(n=n_to_process).reset_index(drop=True)
    selection_mode = (
        f"random sample (seed={RANDOM_SEED})"
        if RANDOM_SEED is not None
        else "random sample"
    )
else:
    df = df_full.head(n_to_process).reset_index(drop=True)
    selection_mode = "first N rows" if n_to_process < total_rows else "all rows"

pipeline = AzureBiasPipeline(
    client,
    AzureBiasPipelineConfig(
        prompt_path=PROMPT_PATH,
        model_for_api=model_for_api,
        temperature=TEMPERATURE,
        chunk_char_limit=CHUNK_CHAR_LIMIT,
        max_retries=MAX_RETRIES,
        sleep_base_sec=SLEEP_BASE_SEC,
        parallel_calls=PARALLEL_CALLS,
        global_max_calls=GLOBAL_MAX_CALLS,
    ),
)

print(f"Input file: {INPUT_CSV}")
print(f"Total rows in file: {total_rows}")
print(f"Rows to process: {n_to_process} ({selection_mode})")
print(f"Output will be saved to: {output_csv}")
print(f"Model: {model_for_api}")
print(f"PARALLEL_CALLS={PARALLEL_CALLS}")
print("-" * 50)

start_time = time.time()
df = pipeline.process_dataframe(df)
output_paths = write_output_bundle(df, output_csv)
total_time = time.time() - start_time

OUTPUT_CSV = output_paths["reviewer_path"]
ANALYSIS_READY_CSV = output_paths["analysis_path"]
REVIEWER_ADJUDICATION_CSV = output_paths["adjudication_path"]
print("-" * 50)
print(f"Reviewer output: {OUTPUT_CSV}")
print(f"Analysis-ready output: {ANALYSIS_READY_CSV}")
print(f"Reviewer adjudication output: {REVIEWER_ADJUDICATION_CSV}")
print(f"Possible bias flags kept: {int(df['Possible_Bias_Count'].sum())}")
print(f"Likely bias flags kept: {int(df['Likely_Bias_Count'].sum())}")
print(f"Total time: {total_time:.1f}s ({total_time / len(df):.1f}s per note)")